# 03 — Querying: the DSL, the reasoner, and θ

Five query shapes over the same cassette store. Each answers a different kind of question and costs different amounts:

| Shape | Call | Cost | When |
|---|---|---|---|
| Index scan | `Query().where(...).run(manifest)` | 0 range gets | List all triples matching a filter |
| Aggregate | `count_by / constraint / first_seen / timeline` | 0 range gets | Summary stats, no hydration |
| Single-claim reasoning | `store.ask(Query(...))` | 1–5 gets | Verify one triple, calibrated verdict |
| Multi-hop | `store.connect(source, target)` | 2–20 gets | Find chain between two entities |
| One-of-many | `store.any_of(source, {t1,...})` | 2–20 gets | Which targets is source connected to? |

**θ is the load-bearing number.** Every verdict carries `(supports, refutes, theta)`. On claims the corpus doesn't speak to, θ → 1.0. This is what keeps the system from confidently hallucinating.

In [ ]:
import json, tempfile, os
from infon.cassette import (
    InfonStore, Query, run_any,
    first_seen, last_seen, timeline, count_by,
)

SCHEMA = {
    # Actors
    "toyota":    {"type": "actor", "tokens": ["toyota"]},
    "honda":     {"type": "actor", "tokens": ["honda"]},
    "ford":      {"type": "actor", "tokens": ["ford"]},
    "tesla":     {"type": "actor", "tokens": ["tesla"]},
    "bmw":       {"type": "actor", "tokens": ["bmw"]},
    "panasonic": {"type": "actor", "tokens": ["panasonic"]},
    "catl":      {"type": "actor", "tokens": ["catl"]},
    # Hierarchy parent (silent anchor)
    "automaker": {"type": "actor", "tokens": ["_none_"]},
    # Relations
    "invest":  {"type": "relation", "tokens": ["invest", "invested", "investment"]},
    "partner": {"type": "relation", "tokens": ["partner", "partners", "partnered", "partnership"]},
    "supply":  {"type": "relation", "tokens": ["supply", "supplies", "supplier"]},
    "acquire": {"type": "relation", "tokens": ["acquire", "acquired", "acquisition"]},
    # Features
    "batteries":   {"type": "feature", "tokens": ["battery", "batteries"]},
    "solid_state": {"type": "feature", "tokens": ["solid-state", "solid state"]},
}

# Parent/child relationships for hierarchy expansion.
for child in ("toyota", "honda", "ford", "tesla", "bmw"):
    SCHEMA[child]["parent"] = "automaker"

tmpdir = tempfile.mkdtemp(prefix="infon_03_")
schema_path = os.path.join(tmpdir, "schema.json")
with open(schema_path, "w") as f:
    json.dump(SCHEMA, f)

store = InfonStore(os.path.join(tmpdir, "store"), schema_path=schema_path)

# Each sentence has at most ONE feature (batteries/solid_state) so the
# actor-object slot isn't contested. This keeps the extraction clean
# enough for multi-hop chains to form — a real corpus will need the
# extraction_report to catch this kind of role competition.
DOCS = [
    {"id": "d01", "timestamp": "2026-01-05",
     "text": "Toyota invested in solid-state technology."},
    {"id": "d02", "timestamp": "2026-01-20",
     "text": "Toyota partnered with Panasonic."},
    {"id": "d03", "timestamp": "2026-02-01",
     "text": "Panasonic supplies CATL."},
    {"id": "d04", "timestamp": "2026-02-15",
     "text": "Honda partnered with CATL."},
    {"id": "d05", "timestamp": "2026-03-01",
     "text": "Tesla invested in batteries."},
    {"id": "d06", "timestamp": "2026-03-15",
     "text": "Ford partnered with BMW."},
    {"id": "d07", "timestamp": "2026-04-05",
     "text": "Toyota no longer partnered with Panasonic."},  # retraction
]

r = store.ingest(DOCS)
print(f"ingested {len(r['ingested'])}  n_infons {r['n_infons']}")
print(r["report"].summary())

## 1. Grammar — pin roles, filter by polarity, confidence

`Query().where(...)` pins triple roles. Multiple `.where()` calls AND together. `.mentioning(a)` is role-free — finds the anchor in any of subject/predicate/object.

All grammar calls return a new `Query` (immutable), so they compose.

In [ ]:
def show(label, hits, limit=6):
    print(f"\n{label}  \u2192 {len(hits)} hits")
    for h in hits[:limit]:
        pol = "\u00ac" if h.loc.polarity == 0 else " "
        print(f"  {pol}{h.loc.subject:>10} {h.loc.predicate:<8} "
              f"{h.loc.object:<14} conf={h.loc.confidence:.2f}")

show("where(subject='toyota')",
     Query().where(subject="toyota").run(store.manifest))

show("where(subject='toyota', predicate='invest')",
     Query().where(subject="toyota", predicate="invest").run(store.manifest))

show("mentioning('catl')\u2003\u2014 role-free lookup",
     Query().mentioning("catl").run(store.manifest))

show("where(subject='toyota').negated()\u2003\u2014 retractions only",
     Query().where(subject="toyota").negated().run(store.manifest))

## 2. Timeline — between, before, after

Timestamp bounds narrow any query to a window. The manifest's per-cassette time bbox lets the pruner skip cassettes that can't contain the range.

In [ ]:
show("where(subject='toyota').between('2026-02-01','2026-03-01')",
     Query().where(subject="toyota")
            .between("2026-02-01", "2026-03-01").run(store.manifest))

show("mentioning('batteries').after('2026-03-01')",
     Query().mentioning("batteries").after("2026-03-01").run(store.manifest))

## 3. Logic — AND (chain), OR (run_any), NOT (contradicting)

Chaining `.where()` gives you AND. For OR, `run_any([q1, q2, ...])` unions the hits across queries. For NOT, `.contradicting()` flips the polarity on a pinned triple — useful for refutation search.

All three live at the index level. Zero hydration.

In [ ]:
# AND — every clause must hold.
show("AND (chain): toyota + invest + after 2026-02-01",
     Query().where(subject="toyota", predicate="invest")
            .after("2026-02-01").run(store.manifest))

# OR — any clause
show("OR: subject=toyota | subject=honda",
     run_any(store.manifest, [
         Query().where(subject="toyota"),
         Query().where(subject="honda"),
     ]))

# NOT — find retractions of a specific claim.
show("contradicting(): refuters of toyota/partner/panasonic",
     Query().where(subject="toyota", predicate="partner",
                    object="panasonic").contradicting().run(store.manifest))

## 4. Aggregates — count_by, first_seen, last_seen, timeline, constraint

The index knows things the reasoner doesn't need to hydrate to find out. `count_by` groups any query by a role. `first_seen` / `last_seen` extract extreme timestamps per anchor. `timeline` gives the full sequence.

`constraint(s, p, o)` is the corpus-level aggregate: how many affirmations, how many refutations, what time span, whether it's contested.

In [ ]:
print("count_by predicate for subject=toyota:")
print(" ", count_by(store.manifest, Query().where(subject="toyota"),
                    groupby="predicate"))

print("\nfirst_seen('panasonic'):", first_seen(store.manifest, "panasonic"))
print("last_seen('toyota'): ",   last_seen(store.manifest, "toyota"))

print("\ntimeline('toyota'):")
for ts, h in timeline(store.manifest, "toyota"):
    pol = "\u00ac" if h.loc.polarity == 0 else " "
    print(f"  {ts}  {pol}{h.loc.subject}/{h.loc.predicate}/{h.loc.object}")

In [ ]:
c = store.constraint("toyota", "partner", "panasonic")
print(f"constraint(toyota/partner/panasonic):")
print(f"  evidence_count: {c.evidence_count}")
print(f"  affirmed:       {c.n_affirmed}")
print(f"  refuted:        {c.n_refuted}")
print(f"  mean_conf:      {c.mean_confidence}")
print(f"  span:           {c.t_min} \u2192 {c.t_max}  ({c.span_days} days)")
print(f"  contested:      {c.is_contested}")
print(f"  balance:        {c.polarity_balance:+.2f}  "
      f"(0 = perfectly split, +1 = all-affirmed, -1 = all-refuted)")

## 5. Hierarchy expansion — parent queries hit descendants

`Query.expand_hierarchy(schema)` walks the schema's parent/child tree so a query pinned to a parent anchor matches hits for every descendant. The manifest pruner still fires over the expanded anchor set.

This lets you ask broad questions ("which automakers invest in batteries?") without enumerating every child.

In [ ]:
# Without expansion: 0 hits because no infon has subject="automaker"
show("where(subject='automaker') \u2014 unexpanded",
     Query().where(subject="automaker").run(store.manifest))

# With expansion: walks to toyota/honda/ford/tesla/bmw
show("where(subject='automaker').expand_hierarchy(schema)",
     Query().where(subject="automaker")
            .expand_hierarchy(store.schema).run(store.manifest))

## 6. Single-claim reasoning — store.ask()

`store.ask(Query(...))` takes a claim, retrieves relevant evidence, computes per-infon Dempster\u2013Shafer masses, combines them via Dempster's rule, and returns a verdict.

**Every verdict cites its sources.** If the GNN and DS rule disagree with the raw text, the sources are still attached — the reasoner can be wrong, but it never lies about what it saw.

In [ ]:
v = store.ask(Query().where(subject="toyota", predicate="invest",
                             object="solid_state"))
print(f"claim:    toyota invests in solid_state")
print(f"verdict:  {v.label}")
print(f"S={v.mass.supports:.2f}  R={v.mass.refutes:.2f}  "
      f"\u03b8={v.mass.theta:.2f}")
print(f"sources ({len(v.sources)}):")
for s in v.sources[:3]:
    print(f"  \u2022 {s.sentence}   (conf={s.confidence:.2f})")

### θ on claims the corpus can't answer

The litmus test: on NEI claims, θ → 1.0, `range_gets == 0`. The pruner short-circuits before any hydration.

In [ ]:
cases = [
    ("toyota/invest/solid_state",  "SUPPORTS"),
    ("ford/acquire/honda",          "NOT_ENOUGH_INFO"),
    ("tesla/partner/catl",          "NOT_ENOUGH_INFO"),
]
for label, expected in cases:
    s, p, o = label.split("/")
    v = store.ask(Query().where(subject=s, predicate=p, object=o))
    mark = "\u2713" if v.label == expected else "\u2717"
    print(f"  {mark} {label:<32} {v.label:<18} "
          f"\u03b8={v.mass.theta:.2f}  gets={v.range_gets}")

## 7. Multi-hop — store.connect()

When the answer requires following a chain of relations, `store.connect(source, target)` runs an MCTS search over the hypergraph. Polarity-aware: a retracted hop at any point turns SUPPORTS into REFUTES.

Connective predicates (partner / supply / acquire / license) are auto-inferred from the corpus structure — no schema annotation needed. Reportive predicates (mention / describe) are filtered out so chains like \`X mentions Y; Y supplies Z\` don't trivially compose.

When `<root>/_model/gnn.pt` exists, the trained sheaf GNN re-scores the MCTS chain as a prior.

In [ ]:
# Auto-inference of connective predicates works well at scale; on a
# 7-doc store it can be flaky. Pass them explicitly when you know them.
CONNECTIVE = {"partner", "supply", "acquire", "license", "invest"}

# toyota → panasonic → catl
v = store.connect("toyota", "catl", connective_predicates=CONNECTIVE)
print(f"toyota → catl: {v.label}  S={v.mass.supports:.2f}  θ={v.mass.theta:.2f}")
for s in v.sources:
    print(f"  {s.subject} → {s.predicate} → {s.object}")

# Disconnected pair
v = store.connect("ford", "catl", connective_predicates=CONNECTIVE)
print(f"\nford → catl:   {v.label}  θ={v.mass.theta:.2f}  (honestly NEI)")

## 8. One tree walk, many targets — store.any_of()

`any_of(source, targets)` resolves connectivity to every target in a single MCTS walk. Cost is roughly `O(1)` in the number of targets: a path passing through an intermediate resolves that intermediate for free.

This is the "which of my suppliers is my competitor connected to?" pattern — pass the whole target set, rank the results.

In [ ]:
vs = store.any_of("toyota", {"catl", "panasonic", "honda", "bmw", "tesla"},
                  connective_predicates=CONNECTIVE)
for t, v in sorted(vs.items(), key=lambda kv: -kv[1].mass.supports):
    print(f"  {t:<10} {v.label:<18}  S={v.mass.supports:.2f}  "
          f"θ={v.mass.theta:.2f}")

---

## Why θ matters

Traditional classifiers output softmax probabilities that always sum to 1 over the label set you gave them. If a claim lies outside that set, the model still commits \u2014 it just picks the least-bad label.

A Dempster\u2013Shafer mass function has a fourth number, θ, that absorbs unassignable evidence. When the system has nothing to say, θ approaches 1 and the verdict becomes `NOT_ENOUGH_INFO`. Not a confident wrong answer.

The cassette pruner reinforces this: if the claim's anchors don't appear in any cassette, *no hydration happens*. θ stays 1.0, `range_gets` stays 0. Ignorance is both honestly reported and cheaply computed.

**Key insight.** The index is the query engine. Hydration is only needed when you want the actual sentence. Most persona questions \u2014 counts, aggregates, timelines, contradiction search \u2014 resolve to 0 range gets.

**Next:**
- [04 — Temporal](04_temporal.ipynb): trajectories and time-travel snapshots.
- [06 — Agent Tools](06_agent_tools.ipynb): the 9-tool `Analyst` that translates natural-language questions into these DSL calls.
- [08 — Category Theory](08_category_theory.ipynb): Kan-based schema migration and sheaf structure.

In [ ]:
import shutil
shutil.rmtree(tmpdir)
print("Done.")